# Machine Learning & Predictive Modeling: Student Performance

**Project:** Student Performance Machine Learning Project  
**Author:** Gabriale McKinley  

## Objective

The goal of this notebook is to build, train, compare, and evaluate predictive machine learning models for a student performance project. The analysis focuses on predicting whether a student is likely to achieve **high academic performance** based on available student characteristics.

This notebook demonstrates the full machine learning workflow, including data preparation, feature engineering, train/test splitting, model comparison, evaluation metrics, hyperparameter tuning, and feature importance analysis.

## Research Question

**Can student performance be predicted using demographic, academic, and behavioral variables?**

For this project, the target variable is converted into a classification outcome:

- **High Performance = 1** if the student's final grade is 70 or higher
- **Low/Moderate Performance = 0** if the student's final grade is below 70

This framing allows the project to evaluate student success as a classification problem.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset

Upload your student performance dataset to the same GitHub repository as this notebook.

The code below assumes the dataset is named:

`student_performance.csv`

If your file has a different name, update the filename in the next code cell.

In [ ]:
df = pd.read_csv('student_performance.csv')

df.head()

## 3. Initial Data Review

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nMissing values:')
print(df.isnull().sum())

print('\nData types:')
print(df.dtypes)

## 4. Create the Target Variable

This project predicts whether a student is likely to reach a high-performance threshold.

Update the `grade_column` below if your dataset uses a different final grade column name. Common examples include:

- `FinalGrade`
- `G3`
- `final_score`
- `Exam_Score`
- `Performance_Index`

In [ ]:
# Change this column name if needed
grade_column = 'FinalGrade'

# Create binary target variable
df['High_Performance'] = np.where(df[grade_column] >= 70, 1, 0)

df[['High_Performance', grade_column]].head()

## 5. Check Class Balance

Class imbalance matters because a model could appear accurate while mostly predicting the majority class. If one group is much larger than the other, class weights will be used during modeling.

In [ ]:
class_counts = df['High_Performance'].value_counts()
print(class_counts)

class_counts.plot(kind='bar')
plt.title('Class Balance: High Performance vs Low/Moderate Performance')
plt.xlabel('Performance Class')
plt.ylabel('Number of Students')
plt.xticks(rotation=0)
plt.show()

## 6. Feature Engineering and Data Preparation

The following steps implement the required data preparation fixes:

- Remove the original final grade column to avoid data leakage
- Separate features from the target variable
- Identify numerical and categorical variables
- Scale numerical variables using `StandardScaler`
- Encode categorical variables using `OneHotEncoder`
- Use an 80/20 train/test split

In [ ]:
# Drop target and original grade column to prevent leakage
X = df.drop(columns=['High_Performance', grade_column])
y = df['High_Performance']

# Identify feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)

## 7. Model Building: Model Tournament

Two classification models are compared:

1. **Logistic Regression**  
   A strong baseline model that is easy to interpret and useful for understanding relationships between features and the target.

2. **Random Forest Classifier**  
   A tree-based ensemble model that can capture nonlinear patterns and interactions between variables.

In [ ]:
logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

random_forest_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

models = {
    'Logistic Regression': logistic_model,
    'Random Forest': random_forest_model
}

## 8. Train and Evaluate Baseline Models

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df

## 9. Confusion Matrices

A confusion matrix shows how many students were correctly and incorrectly classified by each model.

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low/Moderate', 'High'])
    disp.plot()
    plt.title(f'Confusion Matrix: {name}')
    plt.show()
    
    print(f'Classification Report: {name}')
    print(classification_report(y_test, y_pred, target_names=['Low/Moderate', 'High']))

## 10. ROC Curves

The ROC curve helps evaluate how well each model separates high-performing students from low/moderate-performing students across different classification thresholds.

In [ ]:
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test)
    plt.title(f'ROC Curve: {name}')
    plt.show()

## 11. Hyperparameter Tuning

This section tunes the Random Forest model by testing different values for `max_depth`, `n_estimators`, and `min_samples_split`.

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5, 10, None],
    'classifier__min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    random_forest_model,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validated F1 Score:', grid_search.best_score_)

## 12. Evaluate Tuned Random Forest

In [ ]:
best_rf = grid_search.best_estimator_

y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

tuned_results = pd.DataFrame([{
    'Model': 'Tuned Random Forest',
    'Accuracy': accuracy_score(y_test, y_pred_best),
    'Precision': precision_score(y_test, y_pred_best),
    'Recall': recall_score(y_test, y_pred_best),
    'F1 Score': f1_score(y_test, y_pred_best),
    'ROC-AUC': roc_auc_score(y_test, y_prob_best)
}])

pd.concat([results_df, tuned_results], ignore_index=True)

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low/Moderate', 'High'])
disp.plot()
plt.title('Confusion Matrix: Tuned Random Forest')
plt.show()

print(classification_report(y_test, y_pred_best, target_names=['Low/Moderate', 'High']))

## 13. Feature Importance

Feature importance helps explain which variables had the greatest influence on the model's predictions.

In [ ]:
# Get feature names after preprocessing
preprocessor_fitted = best_rf.named_steps['preprocessor']

feature_names = []

if numeric_features:
    feature_names.extend(numeric_features)

if categorical_features:
    encoded_features = preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_features)
    feature_names.extend(encoded_features)

importances = best_rf.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(15)

In [ ]:
top_features = feature_importance_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.gca().invert_yaxis()
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 14. Interpretation Summary

The models were evaluated using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrices. Accuracy alone was not used because it can hide poor performance when the classes are imbalanced.

The Logistic Regression model served as a baseline because it is interpretable and useful for understanding linear relationships between student characteristics and academic performance. The Random Forest model was included because it can capture more complex relationships and interactions between variables.

The better-performing model should be selected based primarily on F1-score, recall, and ROC-AUC rather than accuracy alone. In this project, recall is especially important because failing to identify students at risk of lower performance could prevent schools from offering timely academic support.

The feature importance results show which variables had the strongest influence on predictions. These variables should be interpreted carefully because student performance data may reflect structural factors such as access to resources, prior academic opportunity, attendance patterns, and socioeconomic conditions. The model should be used as a decision-support tool, not as a replacement for educator judgment.

## 15. Conclusion

This machine learning workflow demonstrates how student performance can be predicted using supervised classification models. The project includes appropriate data preparation, train/test separation, model comparison, hyperparameter tuning, professional evaluation metrics, and feature importance analysis.

The results can support early intervention strategies by identifying patterns associated with high or lower academic performance. However, the findings should be used responsibly, with attention to fairness, bias, and the social context behind student data.